In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
sys.path.append(str(Path.cwd().parent))

In [2]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing

from src.preprocessor import TVAEPreprocessorConfig, TVAEPreprocessor
from src.inference import reconstruct, generate_synthetic_samples
from src.train import train_tvae_model, run_latent_dim_sweep
from src.model import TVAEConfig, set_seed

In [ ]:
# Seed once, globally.
set_seed(42)

housing = fetch_california_housing(as_frame=True)
data = housing.frame

In [ ]:
# Preprocess: n_gmm_components=5 matches the notebook (9 cols * (5+1) = 54 = input_dim)
config = TVAEPreprocessorConfig(n_gmm_components=5)
preprocessor = TVAEPreprocessor(config)

all_cols = list(data.columns)
transformed = preprocessor.fit_transform(data, continuous_columns=all_cols, categorical_columns=[])

input_dim = preprocessor.get_output_dim()
output_info = preprocessor.get_output_info()
print(f"input_dim={input_dim}, transformed shape={transformed.shape}")

input_dim=54, transformed shape=(20640, 54)


In [ ]:
# Loaders 80/20 split
dataset = TensorDataset(torch.FloatTensor(transformed))
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(
    dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In [6]:
base_sweep_config = TVAEConfig(
    input_dim=input_dim,
    output_info=output_info,
    num_epochs=30,
    beta=0.5, beta_start=0.0, beta_warmup_epochs=15,
    batch_size=BATCH_SIZE,
    device="cpu",
    dropout_rate=0.1
)

best_latent_dim, sweep_df = run_latent_dim_sweep(
    base_sweep_config, candidate_dims=[4, 8, 12, 16],
    train_loader=train_loader, val_loader=val_loader, verbose=True,
)

Epoch 10/30 - Train Loss: -3.3117 | Val Loss: -4.0411
Epoch 20/30 - Train Loss: -3.6572 | Val Loss: -4.4718
Epoch 30/30 - Train Loss: -3.8355 | Val Loss: -4.6489

Restored best checkpoint (prior_mismatch=0.0346)
latent_dim=4 -> final val_loss=-4.6489
Epoch 10/30 - Train Loss: -3.3881 | Val Loss: -4.3202
Epoch 20/30 - Train Loss: -3.9005 | Val Loss: -5.0442
Epoch 30/30 - Train Loss: -4.0992 | Val Loss: -5.3853

Restored best checkpoint (prior_mismatch=0.1299)
latent_dim=8 -> final val_loss=-5.3853
Epoch 10/30 - Train Loss: -3.0302 | Val Loss: -3.7924
Epoch 20/30 - Train Loss: -3.8023 | Val Loss: -4.8238
Epoch 30/30 - Train Loss: -4.1495 | Val Loss: -5.3051

Restored best checkpoint (prior_mismatch=0.4401)
latent_dim=12 -> final val_loss=-5.3051
Epoch 10/30 - Train Loss: -3.0976 | Val Loss: -3.9973
Epoch 20/30 - Train Loss: -3.8309 | Val Loss: -4.8824
Epoch 30/30 - Train Loss: -4.1210 | Val Loss: -5.2186

Restored best checkpoint (prior_mismatch=0.4635)
latent_dim=16 -> final val_loss=-5

In [7]:
final_config = TVAEConfig(
    input_dim=input_dim,
    output_info=output_info,
    latent_dim=best_latent_dim,
    num_epochs=100,
    beta=1.0, beta_start=1.0, beta_warmup_epochs=0, use_beta_warmup=False,
    batch_size=BATCH_SIZE,
    device="cpu",
    dropout_rate=0.1
)

model, trainer, history = train_tvae_model(final_config, train_loader, val_loader, verbose=True)

Epoch 10/100 - Train Loss: -1.3983 | Val Loss: -1.7418
Epoch 20/100 - Train Loss: -1.5363 | Val Loss: -1.8975
Epoch 30/100 - Train Loss: -1.6243 | Val Loss: -1.9526
Epoch 40/100 - Train Loss: -1.6542 | Val Loss: -2.0155
Epoch 50/100 - Train Loss: -1.7073 | Val Loss: -2.0915
Epoch 60/100 - Train Loss: -1.7299 | Val Loss: -2.1365
Epoch 70/100 - Train Loss: -1.7462 | Val Loss: -2.1593
Epoch 80/100 - Train Loss: -1.7552 | Val Loss: -2.1532
Epoch 90/100 - Train Loss: -1.7675 | Val Loss: -2.1578
Epoch 100/100 - Train Loss: -1.8053 | Val Loss: -2.1511

Restored best checkpoint (prior_mismatch=0.1528)
latent_dim=4 -> final val_loss=-2.1511


In [8]:
reconstructed_transformed = reconstruct(model, transformed, "cpu", output_info)
mse_recon = np.mean((transformed - reconstructed_transformed) ** 2)
reconstructed_data = preprocessor.inverse_transform(reconstructed_transformed)
print(f"VAE Reconstruction MSE: {mse_recon:.6f}")
reconstructed_data.head()

VAE Reconstruction MSE: 14.254684


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,7.266360,48.306009,6.090380,1.035648,808.006059,2.662084,37.753140,-122.195411,3.725468
1,7.199260,26.681985,6.120166,1.032516,943.418502,2.886272,37.719535,-122.091716,3.719456
2,7.280568,48.352979,7.866919,1.035918,802.429139,2.668736,37.756024,-122.214958,3.727332
3,5.218861,48.653338,5.887955,1.034039,815.315266,2.414133,37.738546,-122.254722,3.717658
4,3.687850,48.695092,5.792736,1.038997,786.305607,2.324132,37.761567,-122.271097,3.712276


In [9]:
n_synthetic = len(data)
synthetic_transformed = generate_synthetic_samples(model, n_synthetic, "cpu", output_info)
synthetic_data = preprocessor.inverse_transform(synthetic_transformed)

for col in data.columns:
    print(f"  {col}: Original={data[col].mean():.4f}, Synthetic={synthetic_data[col].mean():.4f}")

  MedInc: Original=3.8707, Synthetic=3.7219
  HouseAge: Original=28.6395, Synthetic=27.9836
  AveRooms: Original=5.4290, Synthetic=5.3744
  AveBedrms: Original=1.0967, Synthetic=1.0730
  Population: Original=1425.4767, Synthetic=1065.0857
  AveOccup: Original=3.0707, Synthetic=2.8345
  Latitude: Original=35.6319, Synthetic=35.6636
  Longitude: Original=-119.5697, Synthetic=-119.5925
  MedHouseVal: Original=2.0686, Synthetic=1.9876


In [10]:
from sdmetrics.reports.single_table import QualityReport, DiagnosticReport
from sdv.metadata import SingleTableMetadata

In [11]:
metadata = SingleTableMetadata()

metadata.detect_from_dataframe(data=data)

print("Detected metadata:")
print(metadata.to_dict())

Detected metadata:
{'columns': {'MedInc': {'sdtype': 'numerical'}, 'HouseAge': {'sdtype': 'numerical'}, 'AveRooms': {'sdtype': 'numerical'}, 'AveBedrms': {'sdtype': 'numerical'}, 'Population': {'sdtype': 'numerical'}, 'AveOccup': {'sdtype': 'numerical'}, 'Latitude': {'pii': True, 'sdtype': 'latitude'}, 'Longitude': {'pii': True, 'sdtype': 'longitude'}, 'MedHouseVal': {'sdtype': 'numerical'}}, 'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1'}


In [12]:
report = QualityReport()
report.generate(data, synthetic_data, metadata.to_dict())

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 9/9 [00:00<00:00, 186.45it/s]|
Column Shapes Score: 77.89%

(2/2) Evaluating Column Pair Trends: |██████████| 36/36 [00:00<00:00, 544.87it/s]|
Column Pair Trends Score: 92.5%

Overall Score (Average): 85.2%



In [13]:
diagnostioc_report = DiagnosticReport()
diagnostioc_report.generate(data, synthetic_data, metadata.to_dict())

Generating report ...

(1/2) Evaluating Data Validity: |██████████| 9/9 [00:00<00:00, 968.07it/s]|
Data Validity Score: 100.0%

(2/2) Evaluating Data Structure: |██████████| 1/1 [00:00<00:00, 761.63it/s]|
Data Structure Score: 100.0%

Overall Score (Average): 100.0%



In [15]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Lasso
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

X_real = data.drop(columns=['MedHouseVal'])
y_real = data['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X_real, y_real, test_size=0.2, random_state=42)

X_syn = synthetic_data.drop(columns=['MedHouseVal'])
y_syn = synthetic_data['MedHouseVal']

models = {
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'Lasso': Lasso(random_state=42),
    'XGB': XGBRegressor(random_state=42)
}

results = []
for name, model_cls in models.items():
    m_real = model_cls.__class__(**model_cls.get_params())
    m_real.fit(X_train, y_train)
    r2_real = r2_score(y_test, m_real.predict(X_test))

    m_syn = model_cls.__class__(**model_cls.get_params())
    m_syn.fit(X_syn, y_syn)
    r2_syn = r2_score(y_test, m_syn.predict(X_test))

    results.append({
        'model': name,
        'r2_real': r2_real,
        'r2_synthetic': r2_syn,
        'utility_ratio': r2_syn / r2_real
    })

utility_df = pd.DataFrame(results)
print(utility_df)
print(f"\nMean utility ratio across models: {utility_df['utility_ratio'].mean():.4f}")

              model   r2_real  r2_synthetic  utility_ratio
0      RandomForest  0.804850      0.580995       0.721867
1  GradientBoosting  0.775645      0.565071       0.728518
2             Lasso  0.284167      0.225254       0.792682
3               XGB  0.830137      0.514588       0.619883

Mean utility ratio across models: 0.7157
